In [110]:
import json
import os
from pydub import AudioSegment
from tqdm import tqdm
import shutil
import pandas as pd
from glob import glob

## Data Cleaning
For each item, we have:
['file', 'source', 'span', 'statement', 'sub', 'split']
We need to do a simple cleaning: if the clip is filled with BG music,

In [154]:
text_json = "./Data/violin_raw_videos/text_data.json"
audio_dir = "./Data/violin_raw_videos/audio_clips"
audio_dir_short = "./Data/violin_raw_videos/audio_clips_part_1_short"

In [4]:
with open(text_json,"r") as f:
    text_data = json.load(f)

In [53]:
def get_long_sub(sub):
    # Concatenate all subtitles
    concatenated_text = " ".join([item[0] for item in sub])

    # Calculate total duration in seconds
    total_duration_ms = sum([end - start for _, (start, end) in sub])
    total_duration_seconds = total_duration_ms / 1000
    # return concatenated_text, total_duration_seconds
    return concatenated_text, total_duration_ms



def segment_audio(audio_path, sub, dest_dir = "./", silence_duration = 500):
    # Load the audio file (FLAC format)
    audio = AudioSegment.from_file(audio_path, format="flac")
    original_name = os.path.splitext(os.path.basename(audio_path))[0]

    silence = AudioSegment.silent(duration=silence_duration)  # Duration in milliseconds

    audio_seg_sub = {}

    # Split audio into segments, add silence, and save them
    for idx, (text, (start, end)) in enumerate(sub):
        clip = audio[start:end]
        clip_with_silence = silence + clip + silence
        segment_filename = os.path.join(dest_dir, f"{original_name}_segment_{idx + 1}.flac")
        clip_with_silence.export(segment_filename, format="flac")
        audio_seg_sub[f"{original_name}_segment_{idx + 1}"] = {
            'file': f"{original_name}_segment_{idx + 1}", 
            'sub': text,
            'duration': end-start+silence_duration*2,
        }
    return audio_seg_sub

## Get a concatenated subtitle json file for all data

In [80]:
new_text = {}
for key in tqdm(text_data.keys(), desc="Processing files"):
    # print(text_data[key]['file'])
    audio_path = os.path.join(audio_dir, f"{text_data[key]['file']}.flac")

    if not os.path.exists(audio_path):
        print(f"Skipping {audio_path}: File does not exist")
        continue  # Skip to the next iteration
    
    audio = AudioSegment.from_file(audio_path, format="flac")
    subtitle, speech_duration = get_long_sub(text_data[key]['sub'])
    new_text[text_data[key]['file']] = {
            'file': text_data[key]['file'], 
            'sub': subtitle,
            'duration': len(audio),
            'speech_duration': speech_duration,
        }
    # print(len(audio)/1000, speech_duration, subtitle)
    # seg_data = segment_audio(audio_path, text_data[key]['sub'], dest_dir=audio_dir_short)
    # new_text_short.update(seg_data)

with open("./Data/violin_raw_videos/data_subtitles.json", "w") as f:
    json.dump(new_text, f, indent=4)

Processing files:  10%|▉         | 1521/15887 [02:25<19:45, 12.12it/s]

Skipping ./Data/violin_raw_videos/audio_clips/8osP7KRacWk_clip_000_040.flac: File does not exist


Processing files:  10%|█         | 1660/15887 [02:39<19:52, 11.93it/s]

Skipping ./Data/violin_raw_videos/audio_clips/IelhpK-eDh0_clip_000_040.flac: File does not exist


Processing files: 100%|██████████| 15887/15887 [41:41<00:00,  6.35it/s]  


## Get part_1 audio data, split them into smaller segments

In [83]:
csv_file = './Data/violin_raw_videos/audio_parts/audio_files_part_1.csv'  # Replace with your CSV file name
df = pd.read_csv(csv_file)

In [122]:
new_text_short = {}
for path in tqdm(df['File Name'], desc="Processing files"):
    name = os.path.splitext(os.path.basename(path))[0]
    if name in text_data.keys():
        # datajj = text_data[name]
        audio_path = os.path.join(audio_dir, f"{text_data[name]['file']}.flac")

        if not os.path.exists(audio_path):
            print(f"Skipping {audio_path}: File does not exist")
            continue  # Skip to the next iteration
        
        seg_data = segment_audio(audio_path, text_data[key]['sub'], dest_dir=audio_dir_short)
        new_text_short.update(seg_data)
    else:
        # print(f"{name} doesn't exist")
        missing.append(name)
with open("./Data/violin_raw_videos/data_subtitles_split_1_short.json", "w") as f:
    json.dump(new_text_short, f, indent=4)

Processing files: 100%|██████████| 2495/2495 [43:54<00:00,  1.06s/it]    


In [ ]:
with open("./Data/violin_raw_videos/data_subtitles_split_1_short.json", "w") as f:
    json.dump(new_text_short, f, indent=4)


In [123]:
# new_text = {}
# new_text_short = {}
# for key in tqdm(text_data.keys(), desc="Processing files"):
#     # print(text_data[key]['file'])
#     audio_path = os.path.join(audio_dir, f"{text_data[key]['file']}.flac")
#     audio = AudioSegment.from_file(audio_path, format="flac")
#     subtitle, speech_duration = get_long_sub(text_data[key]['sub'])
#     new_text[text_data[key]['file']] = {
#             'file': text_data[key]['file'], 
#             'sub': subtitle,
#             'duration': len(audio),
#             'speech_duration': speech_duration,
#         }
#     # print(len(audio)/1000, speech_duration, subtitle)
#     seg_data = segment_audio(audio_path, text_data[key]['sub'], dest_dir=audio_dir_short)
#     new_text_short.update(seg_data)
    

## Clean the data-sub pairs
We have missing text/data.

In [130]:
part1_missing_text = []
part1_missing_audio = []
refined_part1 = {}
for path in tqdm(df['File Name'], desc="Processing files"):
    name = os.path.splitext(os.path.basename(path))[0]
    if name in new_text.keys():
        audio_path = os.path.join(audio_dir, f"{text_data[name]['file']}.flac")
        if not os.path.exists(audio_path):
            print(f"Skipping {audio_path}: File does not exist")
            part1_missing_audio.append(name)
            continue  # Skip to the next iteration
        refined_part1[name]=new_text[name]
    else:
        # print(f"{name} doesn't exist")
        part1_missing_text.append(name)

Processing files: 100%|██████████| 2495/2495 [00:00<00:00, 79299.73it/s]


In [136]:
with open("./Data/violin_raw_videos/data_subtitles_split_1_long.json", "w") as f:
    json.dump(refined_part1, f, indent=4)

In [131]:
part1_missing_audio

[]

In [132]:
len(part1_missing_text)

186

## Not all are in English
Refined from TV Series

In [137]:
new_text

{'gt3ntYidpvs_clip_000_040': {'file': 'gt3ntYidpvs_clip_000_040',
  'sub': 'one board one minute home free okay make it quick you ready yeah wait what are you doing show me superiority the senator dead may drive them back sure sound like a call',
  'duration': 40040,
  'speech_duration': 39840},
 'BgvJlZKbqlw_clip_000_040': {'file': 'BgvJlZKbqlw_clip_000_040',
  'sub': "and the event of my demise are you prepared to lead this family you know jack we're right in the middle of dinner maybe we could talk about this you know when you come into town if I go down Craig I need to know someone will be responsible for the whole Byrnes clan so I ask you Craig are you prepared to be the godfocker the godfocker the godfocker well that is a very powerful turn of phrase jack and when you say godfocker I mean I mean it's I I think I know what you mean but I just what would I do what",
  'duration': 40040,
  'speech_duration': 39750},
 'aazQ5hZrSl8_clip_000_040': {'file': 'aazQ5hZrSl8_clip_000_040',
 

In [ ]:
def is_TV(name):
    result = name.split('_')[0]
    return result in ["friends", "dh", "mf", "himym"]

In [141]:
def refine_dict(original_dict, is_TV):
    return {key: value for key, value in original_dict.items() if is_TV(key)}

In [155]:
new_text_TV = refine_dict(new_text, is_TV)
new_text_part1_short_TV = refine_dict(new_text_short, is_TV)
new_text_part1_long_TV = refine_dict(refined_part1, is_TV)

with open("./Data/violin_raw_videos/data_subtitles_TV.json", "w") as f:
    json.dump(new_text_TV, f, indent=4)

with open("./Data/violin_raw_videos/data_subtitles_split_1_long_TV.json", "w") as f:
    json.dump(new_text_part1_long_TV, f, indent=4)

with open("./Data/violin_raw_videos/data_subtitles_split_1_short_TV.json", "w") as f:
    json.dump(new_text_part1_short_TV, f, indent=4)